# Neural networks through matrix operations

**UE25MA242A: Mathematical Foundation for AI & Data Science**

This notebook explains and inspects the reproducible experiment. Read `docs/REPORT.md` for the complete derivation. It imports the shared NumPy implementation. Run all cells from the project or notebooks directory.

Course inspiration: supplied Projects 10 (transformations), 11 (PCA) and 14 (SVD). Neural networks are an extension of these topics.


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'src/neural_network.py').exists():
    raise RuntimeError('Open this notebook from neural_network_LA_project or its notebooks directory.')
sys.path.insert(0, str(ROOT / 'src'))
from neural_network import NeuralNetwork, spiral_data, stratified_split, gradient_check, spectral_analysis, separation_witness
from run_project import run


## 1. Reproducible data

Samples are columns: $X\in\mathbb{R}^{2\times 400}$. Each spiral has two turns and Cartesian noise standard deviation 0.0125. The disjoint stratified split has 240 training, 80 validation and 80 test points.


In [ ]:
X, Y = spiral_data(seed=42)
train, validation, test = stratified_split(Y, seed=2026)
assert len(set(np.concatenate([train, validation, test]))) == 400
print(X.shape, Y.shape, [len(i) for i in (train, validation, test)])
print('Convex hull overlap witness:', separation_witness(X,Y))


## 2. Shared forward and backward implementation

$Z_l=W_l A_{l-1}+b_l\mathbf{1}^{T}$, hidden $A_l=\max(0,Z_l)$ and output $P=\mathrm{softmax}(Z_3)$. The 2-16-16-2 model has $48+272+34=354$ parameters.

For mean cross-entropy, use unaveraged $D_3=P-Y$, $D_l=(W_{l+1}^{T}D_{l+1})\odot\mathbf{1}_{Z_l>0}$, $\nabla W_l=D_l A_{l-1}^{T}/m$, $\nabla b_l=D_l\mathbf{1}/m$. Averaging occurs exactly once.


In [ ]:
model = NeuralNetwork(seed=42)
P, cache = model.forward(X[:,train])
grads = model.backward(Y[:,train], cache)
print('Parameters:', model.parameter_count)
print('Gradient shapes:', {k:v.shape for k,v in grads.items()})
small = NeuralNetwork((2,4,3,2), seed=17)
print('Centered finite differences:', gradient_check(small, X[:,train[:5]], Y[:,train[:5]]))


## 3. Load or regenerate the declared experiment

By default use the saved results. Set `RETRAIN=True` to run all nine trainings with the declared seeds and budget. Checkpoint selection uses validation loss, and test predictions have no effect on selection.


In [ ]:
RETRAIN = False
if RETRAIN or not (ROOT / 'results/metrics.json').exists():
    run(ROOT / 'results')
metrics = json.loads((ROOT / 'results/metrics.json').read_text())
for row in metrics['runs']:
    print(row['model'], 'seed', row['seed'], 'epoch', row['selected_epoch'],
          'train', round(row['train']['accuracy'],4),
          'validation', round(row['validation']['accuracy'],4),
          'test', round(row['test']['accuracy'],4))
with np.load(ROOT / 'results/weights_deep_42.npz') as saved:
    model.params = {k:saved[k].copy() for k in saved.files}
print('Restored test evaluation:', model.evaluate(X[:,test],Y[:,test]))


In [ ]:
for filename in ('model_comparison.png','learning_curves.png','confusion_matrix.png'):
    display(Image(filename=str(ROOT / 'results/figures' / filename)))


## 4. PCA and SVD answer different questions

PCA fits a basis to **centered training activations** and projects test activations with that basis. A two-dimensional plot may lose classification information. Weight SVD describes a single linear map before ReLU. It cannot establish neuron activity, calibration or whole-network invertibility.

For a rectangular $2\times16$ output matrix, full row rank is 2 and input nullity is 14. Gram eigenvalues include structural zeros.


In [ ]:
for d in spectral_analysis(model):
    print('Layer',d['layer'], 'shape',d['shape'], 'rank',d['rank'],
          'input nullity',d['input_nullity'], 'nonzero-spectrum ratio',d['condition_nonzero_spectrum'])
    assert d['gram_identity_max_error'] < 1e-10
    assert d['svd_reconstruction_relative_error'] < 1e-10
display(Image(filename=str(ROOT / 'results/figures/hidden_representations.png')))
display(Image(filename=str(ROOT / 'results/figures/singular_values.png')))


## 5. SVD rank approximation

$W_{2,k}=U_k\Sigma_kV_k^T$ and $\|W_2-W_{2,k}\|_F^2=\sum_{i>k}\sigma_i^2$. This guarantees optimal matrix approximation at rank k in Frobenius norm, not preserved classifier accuracy. The factorization uses 32k entries, compared with 256 for dense W2. The supplied results materialize W2_k and do not save compressed factors.


In [ ]:
U,s,Vt = np.linalg.svd(model.params['W2'],full_matrices=False)
for row in metrics['rank_truncation']:
    k = row['rank']
    Wk = (U[:,:k]*s[:k]) @ Vt[:k]
    error = np.linalg.norm(model.params['W2']-Wk,'fro')
    assert np.isclose(error,np.linalg.norm(s[k:]),atol=1e-12)
    print('rank',k, 'energy',round(row['weight_energy_retained'],4),
          'factor entries',row['factor_entries'], 'test',row['test']['accuracy'])


## Conclusions and limits

The experiment supports the benefit of nonlinear features under this configuration. It does not prove that depth is mandatory or that linear classifiers are limited to 50%. One hidden nonlinear layer can be expressive at sufficient width. Three initialization seeds and one synthetic split are a limited empirical study.

References: [Deep Learning, Chapter 6](https://www.deeplearningbook.org/contents/mlp.html), [Stanford gradient checks](https://cs231n.github.io/neural-networks-3/), [NumPy SVD](https://numpy.org/doc/stable/reference/generated/numpy.linalg.svd.html), and the supplied references preserved in the workspace.
